# 02 · Preparación de los datos — SportSmart Perú

Fase CRISP-DM: **Preparación de los datos**. Aplica los 23 tratamientos del informe de calidad (`docs/02_datos/informe_calidad.md`) e integra las tablas. `data/raw` no se modifica; las tablas limpias quedan en `data/processed`.

In [1]:
import sys, re, json
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]  # notebooks/02_preparation -> raíz
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.config import DOCS, REPORTS
from src.features import preparacion
pd.set_option('display.max_colwidth', 70)
res = preparacion.main()

Incidencias: 5032 -> 1147. Tablas en data/processed


## 1. Controles antes y después

Cada problema se mide con el mismo control sobre los datos crudos y sobre los limpios.

In [2]:
informe = (DOCS / '02_datos' / 'informe_calidad.md').read_text(encoding='utf-8')
nombres = {m[0]: m[1].replace('`', '').strip() for m in re.findall(r'^\| (Q\d\d) \| ([^|]+) \|', informe, flags=re.M)}
tabla = pd.DataFrame(res['controles']).T[['antes', 'despues']].assign(problema=lambda x: x.index.map(nombres))[['problema', 'antes', 'despues']]
print('Incidencias:', res['incidencias_antes'], '->', res['incidencias_despues'])
tabla

Incidencias: 5032 -> 1147


,problema,antes,despues
Q01,pedidos.fecha_pedido (fecha 2030-01-01),25.0,0.0
Q02,clientes.departamento mal escrito,752.0,0.0
Q03,clientes.genero no estándar,580.0,0.0
Q04,clientes filas duplicadas,45.0,0.0
Q05,detalle_pedido filas duplicadas,120.0,0.0
Q06,pedidos.monto_total nulo,60.0,0.0
Q07,detalle_pedido.cantidad ≤ 0,20.0,0.0
Q08,detalle_pedido.cantidad > 10 (hasta 99),20.0,0.0
Q09,pagos.importe no cuadra con el pedido,80.0,0.0
Q10,clientes.edad nula,901.0,0.0


## 2. Coherencia entre tablas

Controles sobre los datos limpios. Todos deben dar 0.

In [3]:
pd.Series(res['coherencia'], name='casos').to_frame()

,casos
monto_distinto_a_detalle,0
importe_distinto_a_cantidad_por_precio,0
envio_antes_de_pedido,0
opinion_antes_de_pedido,0
unidades_inventario_distintas_a_ventas,0
balance_inventario_no_cuadra,0
pedidos_fuera_de_periodo_validos,0


## 3. Filas y ventas

In [4]:
pd.DataFrame(res['filas']).T.assign(eliminadas=lambda x: x.antes - x.despues)

,antes,despues,eliminadas
clientes,15045,15000,45
productos,321,321,0
pedidos,63413,63413,0
detalle_pedido,103047,102927,120
pagos,63413,63413,0
campanias,20,20,0
marketing,96056,96056,0
navegacion,32808,32808,0
atencion,5707,5707,0
opiniones,22911,22911,0


In [5]:
pd.Series(res['ventas_validas'], name='valor').to_frame()

,valor
pedidos_antes,60216.00
pedidos_despues,60300.00
soles_antes,13968141.92
soles_despues,13988027.87
diferencia_soles,19885.95


## 4. Vistas integradas

- `ventas_items`: una fila por ítem vendido, con datos del pedido, producto y cliente.
- `pedidos_integrados`: una fila por pedido, con cliente, pago y envío.

In [6]:
pd.DataFrame(res['vistas_integradas']).T

,filas,columnas
ventas_items,102927,26
pedidos_integrados,63413,28


## 5. Conclusiones

- Las incidencias bajan de 5 032 a 1 147.
- De las que quedan, 1 145 son comentarios vacíos que se conservan a propósito (su puntuación sigue siendo útil) y 2 corresponden a un registro de inventario con dos errores a la vez, que no se puede reconstruir.
- Los 7 controles de coherencia dan 0. Las unidades del inventario coinciden exactamente con las ventas por producto y mes.
- Las ventas válidas suben S/ 19 885,95 y se recuperan 84 pedidos, como anticipaba el informe de calidad.
- El informe de calidad reporta 5 031 incidencias porque cuenta una vez cada fila con balance incorrecto; aquí se cuentan por separado los dos errores del registro que tiene ambos.

Las imputaciones de edad y días de entrega quedan marcadas con banderas para rehacerlas solo con datos de entrenamiento en los modelos.